<a href="https://colab.research.google.com/github/Napatphorn/GE338-Lab4/blob/main/Lab4_Modeling_6606614730.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab4_Modeling นักศึกษาเลือกที่จะทำโมเดล การวิเคราะห์พื้นที่เสี่ยงภัยดินถล่ม

In [ ]:
# 1. ติดตั้งไลบรารี geemap สำหรับแสดงแผนที่
!pip install geemap -q

import ee
import geemap

ee.Authenticate()
ee.Initialize(project='v6-dk-459909')

In [ ]:
# 1. สร้างหน้าต่างแผนที่
Map = geemap.Map()

# 2. ดึงข้อมูลขอบเขตการปกครอง "ระดับจังหวัด" (Level 1) ของทั้งโลก
provinces = ee.FeatureCollection("FAO/GAUL/2015/level1")

# 3. กรองเอาเฉพาะจังหวัด "เชียงใหม่"
chiang_mai = provinces.filter(ee.Filter.eq('ADM1_NAME', 'Chiang Mai'))

# 4. ปรับมุมมองแผนที่ให้ซูมพอดีกับจังหวัดเชียงใหม่
Map.centerObject(chiang_mai, 8)

# 5. แสดงผลขอบเขตเชียงใหม่
# (กำหนด style ให้เส้นขอบเป็นสีแดง และข้างในโปร่งใส)
style = {'color': 'red', 'fillColor': '00000000', 'width': 2}
Map.addLayer(chiang_mai.style(**style), {}, 'Chiang Mai Boundary')

# 6. เรียกคำสั่งแสดงแผนที่ออกมาบนหน้าจอ Colab
Map

ปัจจัยที่ 1: ความลาดชัน (Slope)

เมื่อระดับความลาดชันของพื้นที่เพิ่มขึ้น แรงโน้มถ่วงที่กระทำต่อมวลดินจะมีค่าสูงขึ้น ในขณะที่แรงต้านทานการเลื่อนไถลของดินจะลดลง ส่งผลให้พื้นที่ที่มีความลาดชันสูงมีความเปราะบางและเสี่ยงต่อการพังทลายของมวลดิน


In [ ]:
# 1. ดึงข้อมูลความสูงภูมิประเทศ (DEM) จาก SRTM และตัดตามขอบเขตเชียงใหม่
dem = ee.Image("USGS/SRTMGL1_003").clip(chiang_mai)

# 2. คำนวณความลาดชันจาก DEM
slope = ee.Terrain.slope(dem)

# 3. กำหนดสีสำหรับแสดงผล (จาก 0 องศา ถึง 60 องศา)
# สีเขียวเข้ม = พื้นราบ, สีเหลือง = เนินเขา, สีส้ม/แดง = ภูเขาสูงชัน
slope_viz = {
    'min': 0,
    'max': 60,
    'palette': ['006600', 'ddee00', 'ff9900', 'ff0000']
}

# 4. เพิ่มชั้นข้อมูลความลาดชันลงไปในแผนที่
Map.addLayer(slope, slope_viz, 'Factor 1: Slope')

# 5. เรียกแสดงแผนที่อีกครั้งเพื่อดูผลลัพธ์ที่อัปเดต
Map

ปัจจัยที่ 2: ปริมาณน้ำฝนสะสม (Rainfall)

ปริมาณน้ำฝนเป็นปัจจัยกระตุ้นที่ทำให้เกิดดินถล่ม เมื่อมีปริมาณฝนตกสะสม น้ำจะซึมผ่านผิวดินลงสู่ชั้นดินด้านล่าง น้ำฝนจะไปลดแรงเสียดทานและการยึดเกาะระหว่างเม็ดดิน เมื่อดินอุ้มน้ำจนถึงจุดอิ่มตัวและทนรับน้ำหนักที่เพิ่มขึ้นไม่ไหว จึงเกิดการเลื่อนไถลลงมา



In [ ]:
# 1. ดึงข้อมูลฝนจาก ERA5 เฉพาะช่วงฤดูฝน (พ.ค. - ต.ค. 2023)
rain = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR") \
    .filterDate('2023-05-01', '2023-10-31') \
    .select('total_precipitation_sum') \
    .sum() \
    .clip(chiang_mai)

# 2. กำหนดสีแสดงผลปริมาณฝนสะสม
# สีเขียว/ฟ้าอ่อน = ฝนน้อย, สีน้ำเงิน/น้ำเงินเข้ม = ฝนตกหนักมาก
rain_viz = {
    'min': 0.5,
    'max': 2.5,
    'palette': ['#e0f3db', '#a8ddb5', '#4eb3d3', '#08589e']
}

# 3. เพิ่มชั้นข้อมูลปริมาณน้ำฝนลงไปในแผนที่
Map.addLayer(rain, rain_viz, 'Factor 2: Rainfall')

# 4. เรียกแสดงแผนที่เพื่อดูผลลัพธ์
Map

ปัจจัยที่ 3: ความหนาแน่นของพืชพรรณ (NDVI)
ดัชนีพืชพรรณถูกนำมาใช้เป็นตัวแทนของสภาพการปกคลุมดิน ระบบรากของต้นไม้ จะช่วยยึดเหนี่ยวและตรึงชั้นดินไว้ด้วยกัน นอกจากนี้ เรือนยอดของต้นไม้ยังช่วยลดพลังงานจลน์ของเม็ดฝนที่ตกกระทบผิวดิน และชะลออัตราการไหลบ่าของน้ำผิวดิน (Surface runoff) ดังนั้น พื้นที่ที่มีค่า NDVI ต่ำเช่น พื้นที่เปิดโล่ง พื้นที่เกษตรกรรม จึงมีความเสี่ยงต่อการเกิดดินถล่มสูงกว่าพื้นที่ที่มีป่าไม้ปกคลุมหนาแน่น

In [ ]:
# 1. ดึงข้อมูล NDVI จาก MODIS ตลอดปี 2023
ndvi = ee.ImageCollection("MODIS/061/MOD13Q1") \
    .filterDate('2023-01-01', '2023-12-31') \
    .select('NDVI') \
    .median() \
    .multiply(0.0001) \
    .clip(chiang_mai)

# 2. กำหนดสีสำหรับแสดงผล
# สีแดง = พืชพรรณน้อย (หน้าดินเปิดโล่ง/เสี่ยงสูง)
# สีเหลือง = พืชพรรณปานกลาง
# สีเขียว = ป่าไม้หนาแน่น (ปลอดภัย/เสี่ยงต่ำ)
ndvi_viz = {
    'min': 0.2,
    'max': 0.8,
    'palette': ['red', 'yellow', 'green']
}

# 3. เพิ่มชั้นข้อมูล NDVI ลงไปในแผนที่
Map.addLayer(ndvi, ndvi_viz, 'Factor 3: NDVI')

# 4. เรียกแสดงแผนที่เพื่อดูผลลัพธ์
Map

ปัจจัยที่ 4: ระยะห่างจากแหล่งน้ำ (Distance to Water)
 พื้นที่ที่ตั้งอยู่ใกล้กับทางน้ำมักจะได้รับผลกระทบจากการกัดเซาะของกระแสน้ำบริเวณฐานของตลิ่งหรือเชิงเขา การสูญเสียมวลดินบริเวณฐานรากนี้จะทำให้ดินส่วนบนสูญเสียแรงพยุงตัว นอกจากนี้ ชั้นดินบริเวณใกล้ทางน้ำมักจะมีความชื้นสะสมสูง ทำให้มีแนวโน้มที่จะเข้าสู่สภาวะอิ่มตัวได้เร็วกว่าพื้นที่อื่นเมื่อเกิดฝนตกหนัก พื้นที่ที่อยู่ใกล้แหล่งน้ำจึงถูกจัดให้อยู่ในกลุ่มที่มีความเสี่ยงสูง

In [ ]:
# 1. ดึงข้อมูลแหล่งน้ำผิวดินจาก JRC Global Surface Water
water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater") \
    .select('occurrence') \
    .gt(0) \
    .clip(chiang_mai)

# 2. คำนวณระยะห่างจากแหล่งน้ำ (จำกัดรัศมีการค้นหาที่ 3,000 เมตร เพื่อลดภาระการประมวลผล)
water_dist = water.distance(ee.Kernel.euclidean(3000, 'meters')).clip(chiang_mai)

# 3. กำหนดสีสำหรับแสดงผล
# สีน้ำเงินเข้ม = ใกล้น้ำมาก (0 เมตร / เสี่ยงสูง)
# สีฟ้าอ่อน -> ขาว = ไกลน้ำ (3,000 เมตร / ปลอดภัยกว่า)
water_dist_viz = {
    'min': 0,
    'max': 3000,
    'palette': ['08306b', '4292c6', 'c6dbef', 'ffffff']
}

# 4. เพิ่มชั้นข้อมูลระยะห่างจากแหล่งน้ำลงไปในแผนที่
Map.addLayer(water_dist, water_dist_viz, 'Factor 4: Distance to Water')

# 5. เรียกแสดงแผนที่เพื่อดูผลลัพธ์
Map

รวมน้ำหนัก (Weighted Overlay)


In [ ]:
# ==========================================
# 1. เติม .unmask(3000)
# ==========================================
water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('occurrence').gt(0).clip(chiang_mai)

water_dist = water.distance(ee.Kernel.euclidean(3000, 'meters')).unmask(3000).clip(chiang_mai)

# ==========================================
# 2. ปรับสเกลและรวมน้ำหนักใหม่ทั้งหมด
# ==========================================
def normalize(image, min_val, max_val):
    return image.subtract(min_val).divide(ee.Number(max_val).subtract(min_val)).clamp(0, 1)

# ปรับสเกล
slope_norm = normalize(slope, 0, 60)
rain_norm = normalize(rain, 0.5, 2.5)
ndvi_norm = normalize(ndvi, 0, 1)
water_dist_norm = normalize(water_dist, 0, 3000)

# กลับค่า
ndvi_inv = ee.Image(1).subtract(ndvi_norm)
water_dist_inv = ee.Image(1).subtract(water_dist_norm)

# รวมน้ำหนัก
landslide_risk = slope_norm.multiply(0.4) \
    .add(rain_norm.multiply(0.3)) \
    .add(ndvi_inv.multiply(0.15)) \
    .add(water_dist_inv.multiply(0.15)) \
    .rename('Risk_Score')

# ==========================================
# 3. แสดงผล (ปรับช่วงสีให้ดึงสีแดง/เหลืองออกมา)
# ==========================================
risk_viz = {
    'min': 0.15,
    'max': 0.5,
    'palette': ['blue', 'cyan', 'green', 'yellow', 'red']
}

Map.addLayer(landslide_risk, risk_viz, 'Landslide Risk Score (Fixed)')

# เรียกแสดงแผนที่
Map

จะได้เป็น แผนที่ความเสี่ยงดินโคลนถล่ม


In [ ]:
# 1. กำหนดชื่อและสีของแต่ละระดับความเสี่ยงให้ตรงกับแผนที่
legend_dict = {
    'Very High Risk (เสี่ยงสูงมาก)': 'FF0000', # สีแดง
    'High Risk (เสี่ยงสูง)': 'FFFF00',      # สีเหลือง
    'Moderate Risk (เสี่ยงปานกลาง)': '008000', # สีเขียว
    'Low Risk (เสี่ยงต่ำ)': '00FFFF',       # สีฟ้า
    'Very Low Risk (ปลอดภัย)': '0000FF'   # สีน้ำเงิน
}

# 2. นำ Legend ไปแปะไว้บนแผนที่
Map.add_legend(title="Landslide Risk Level", legend_dict=legend_dict)

# 3. เรียกแสดงแผนที่อีกครั้ง (คราวนี้จะมีกล่อง Legend โผล่มาที่มุมขวาล่าง)
Map

คำนวณว่าจังหวัดเชียงใหม่มีพื้นที่เสี่ยงดินถล่มกี่ตารางกิโลเมตร

In [ ]:
# ==========================================
# การคำนวณพื้นที่เสี่ยงดินถล่มระดับสูง (High Risk Area Calculation)
# ==========================================

# 1. คัดกรองเอาเฉพาะพื้นที่ที่คะแนนความเสี่ยงมากกว่า 0.4 (เสี่ยงสูง - เสี่ยงสูงมาก)
high_risk_mask = landslide_risk.gt(0.4)

# 2. สร้างชั้นข้อมูลพื้นที่ (หน่วยเป็นตารางเมตร) แล้วตัดเอาเฉพาะจุดที่ตรงกับเงื่อนไขเสี่ยงสูง
high_risk_area_img = ee.Image.pixelArea().updateMask(high_risk_mask)

# 3. สั่งคำนวณผลรวมพื้นที่ทั้งหมดที่อยู่ในขอบเขตจังหวัดเชียงใหม่
print("กำลังคำนวณพื้นที่ โปรดรอสักครู่...")
high_risk_stats = high_risk_area_img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=chiang_mai.geometry(),
    scale=1000,
    maxPixels=1e10
)

# 4. ดึงค่าตัวเลขออกมาแปลงจาก ตารางเมตร (sq.m.) ให้เป็น ตารางกิโลเมตร (sq.km.)
area_sqm = high_risk_stats.get('area').getInfo()
area_sqkm = area_sqm / 1000000

# 5. แสดงผลลัพธ์
print("==================================================")
print(f"ผลการวิเคราะห์พื้นที่เสี่ยงภัยดินโคลนถล่ม จังหวัดเชียงใหม่")
print("==================================================")
print(f"พื้นที่เสี่ยงระดับสูง-สูงมาก (> 0.4): {area_sqkm:,.2f} ตารางกิโลเมตร")

เปรียบเทียบดินถล่มที่เกิดขึ้นจริงกับโมเดล

In [ ]:
# ==========================================
# ภารกิจที่ 4: Validation (Visual Inspection Approach)
# ==========================================

# 1. คัดกรองเฉพาะพื้นที่ที่มีความเสี่ยงสูงมาก (Threshold > 0.45) เพื่อหาจุดตรวจสอบ
very_high_risk = landslide_risk.updateMask(landslide_risk.gt(0.45))

# 2. เพิ่มชั้นข้อมูลภาพถ่ายดาวเทียมความละเอียดสูง (Satellite Hybrid) เพื่อใช้ดูรอยดินถล่ม
Map.add_basemap('HYBRID')

# 3. แสดงชั้นข้อมูลความเสี่ยงทับบนภาพถ่ายดาวเทียม
Map.addLayer(very_high_risk, {'palette': ['red']}, 'Validation: Very High Risk Areas')

# 4. ซูมไปที่บริเวณที่มีความเสี่ยงสูง
Map.setCenter(98.9, 19.2, 12)

print("คำแนะนำการ Validation:")
print("1. ให้มองหาพื้นที่สีแดง (เสี่ยงสูง) ที่อยู่บนภาพถ่ายดาวเทียม")
print("2. ปิด/เปิด Layer 'Very High Risk Areas' เพื่อดูลักษณะภูมิประเทศจริงด้านล่าง")
print("3. ตรวจสอบว่าพื้นที่สีแดงตรงกับ 'ทางน้ำ' 'ร่องเขา' หรือ 'พื้นที่ที่มีรอยดินขาด' หรือไม่")

Map